# 08 SQL cross-checks

| Field | Value |
|---|---|
| **Purpose** | Recompute key results in SQL from row-level data and compare them with the Python pipeline |
| **Assignment requirement** | **SQL** in the analytics toolset (Technology and Tools) |
| **Inputs** | `data/processed/job_market.sqlite` (written by the pipeline's `database` step from `sql/schema.sql`), the queries `sql/01_*.sql` - `sql/08_*.sql`, and the pipeline tables they are compared with |
| **Outputs** | None |
| **Build identity** | Printed by the first code cell |

**What this shows.** SQL independently reproduces selected aggregations and evaluation metrics from the pipeline's verified row-level outputs. The database holds row-level tables only: one row per job (cleaned fields), one category prediction per job, the human labels, and each recommendation method's ranked results. Every query starts from those rows and never reads a summary table.

**What it does not show.** It does not verify the whole project: the cleaned fields, the taxonomy predictions and the recommender rankings themselves come from the Python pipeline and are inputs here.

**Comparison rule (decision D-037).** Counts must be **exactly** equal. The pipeline's CSV tables store decimals rounded to 6 places, so an exact SQL value can differ from the stored value by at most half a unit in the 6th place; a decimal passes when the difference is at most 0.0000005 + 1e-9. Medians, counts and labels are compared without any allowance.

In [1]:
from job_market.notebook import start

project = start()

RUN_ID: 20261005T080528Z-90e95d23f3
Source: data/raw/all_upwork_jobs_2024-02-07-2024-03-24.csv (244,828 rows)
Taxonomy version: 1.0
Verified outputs: 53 file(s), all hashes match
Environment: matches environment.yml


In [2]:
import pandas as pd

from job_market.database import build_info, compare, matches_build, query_files, run_query

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)

database = project.processed("job_market.sqlite")          # SHA-256 verified against the build
info = build_info(database)
queries = {path.name[:2]: path for path in query_files(project.settings.root)}
print(f"Database schema {info['schema_version']}; input fingerprints match this build: "
      f"{matches_build(info, project.manifest)}")
print("Queries:", ", ".join(path.name for path in queries.values()))

results = {}


def show_query(number):
    text = queries[number].read_text(encoding="utf-8")
    print(text)
    return text


def record(number, table):
    results[number] = table
    passed = int(table["passed"].sum())
    print(f"Q{number}: {passed} of {len(table)} compared values pass"
          + ("" if table.empty or table["difference"].dropna().empty
             else f"; largest absolute difference {table['difference'].abs().max():.2e}"))
    return table[~table["passed"]] if (~table["passed"]).any() else "PASS"

Database schema 1; input fingerprints match this build: True
Queries: 01_reliable_window.sql, 02_monthly_volume.sql, 03_monthly_category_share.sql, 04_monthly_country_share.sql, 05_weekly_hourly_share.sql, 06_country_median_hourly_rate.sql, 07_category_label_accuracy.sql, 08_recommender_ndcg_precision.sql


In [3]:
counts = run_query(database, "SELECT (SELECT COUNT(*) FROM jobs) AS jobs, (SELECT COUNT(*) FROM job_categories) AS categories, "
                             "(SELECT COUNT(*) FROM category_labels) AS category_labels, "
                             "(SELECT COUNT(*) FROM relevance_labels) AS relevance_labels, "
                             "(SELECT COUNT(*) FROM recommendation_results) AS recommendation_results")
counts

,jobs,categories,category_labels,relevance_labels,recommendation_results
0,244828,244828,500,122,180


## Q1. The reliable time-series window

In [4]:
q1 = run_query(database, show_query("01"))
q1

-- Q1. How many days and jobs are in the reliable time-series window?
SELECT MIN(posted_date)            AS first_day,
       MAX(posted_date)            AS last_day,
       COUNT(DISTINCT posted_date) AS days,
       COUNT(*)                    AS jobs
FROM jobs
WHERE eligible_for_timeseries = 1;



,first_day,last_day,days,jobs
0,2024-02-16,2024-03-23,37,229520


In [5]:
reference = pd.DataFrame([{"first_day": project.counts["timeseries_window_start"],
                           "last_day": project.counts["timeseries_window_end"],
                           "days": project.counts["window_days"], "jobs": project.counts["window_jobs"]}])
reference["key"] = q1["key"] = "window"
check = compare(q1, reference, ["key", "first_day", "last_day"], ["days", "jobs"], [])
record("01", check)

Q01: 2 of 2 compared values pass; largest absolute difference 0.00e+00


'PASS'

## Q2. Jobs per day by month (plain and weekday-standardized)

In [6]:
q2 = run_query(database, show_query("02"))
q2

-- Q2. Jobs per day by calendar month: plain and weekday-standardized.
-- Weekday-standardized = the average of the seven weekday means, so a month
-- with more Fridays or Saturdays is not favoured (NULL if a weekday is missing).
WITH daily AS (
    SELECT substr(posted_date, 1, 7)                  AS month,
           posted_date,
           CAST(strftime('%w', posted_date) AS INTEGER) AS weekday,
           COUNT(*)                                   AS jobs
    FROM jobs
    WHERE eligible_for_timeseries = 1
    GROUP BY posted_date
),
weekday_means AS (
    SELECT month, weekday, AVG(jobs) AS mean_jobs
    FROM daily
    GROUP BY month, weekday
),
standardized AS (
    SELECT month, CASE WHEN COUNT(*) = 7 THEN AVG(mean_jobs) END AS jobs_per_day_weekday_std
    FROM weekday_means
    GROUP BY month
)
SELECT d.month,
       MIN(d.posted_date)         AS first_day,
       MAX(d.posted_date)         AS last_day,
       COUNT(*)                   AS days_observed,
       SUM(d.jobs)     

,month,first_day,last_day,days_observed,jobs,jobs_per_day,jobs_per_day_weekday_std
0,2024-02,2024-02-16,2024-02-29,14,88871,6347.928571,6347.928571
1,2024-03,2024-03-01,2024-03-23,23,140649,6115.173913,6170.785714


In [7]:
reference = project.read_csv("monthly_market_summary.csv")
record("02", compare(q2, reference, ["month", "first_day", "last_day"], ["days_observed", "jobs"],
                     ["jobs_per_day", "jobs_per_day_weekday_std"]))

Q02: 8 of 8 compared values pass; largest absolute difference 4.29e-07


'PASS'

## Q3. Category share by month

In [8]:
q3 = run_query(database, show_query("03"))
q3.head(8)

-- Q3. Share of each month's jobs by job function (category).
SELECT substr(j.posted_date, 1, 7) AS month,
       c.primary_category,
       COUNT(*)                    AS jobs,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY substr(j.posted_date, 1, 7)) AS share_pct
FROM jobs AS j
JOIN job_categories AS c USING (row_id)
WHERE j.eligible_for_timeseries = 1
GROUP BY month, c.primary_category
ORDER BY month, jobs DESC, c.primary_category;



,month,primary_category,jobs,share_pct
0,2024-02,web_development,12644,14.227363
1,2024-02,other_unclear,11031,12.412373
2,2024-02,design_creative,10149,11.419923
3,2024-02,digital_marketing,9063,10.197927
4,2024-02,software_it,6553,7.373609
5,2024-02,video_animation,5749,6.468927
6,2024-02,writing_content,4865,5.474227
7,2024-02,admin_virtual_assistance,4109,4.623555


In [9]:
reference = project.read_csv("monthly_category_share.csv")
record("03", compare(q3, reference, ["month", "primary_category"], ["jobs"], ["share_pct"]))

Q03: 92 of 92 compared values pass; largest absolute difference 4.91e-07


'PASS'

## Q4. Client-country share by month (every known country)

In [10]:
q4 = run_query(database, show_query("04"))
print(f"{len(q4)} month-country rows; the validation covers all of them, not only the top countries")
q4.head(8)

-- Q4. Share of each month's jobs with a known client country, for every country.
-- A missing country is excluded, never counted as a country of its own.
SELECT substr(posted_date, 1, 7) AS month,
       country,
       COUNT(*)                  AS jobs,
       100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY substr(posted_date, 1, 7)) AS share_pct
FROM jobs
WHERE eligible_for_timeseries = 1
  AND country IS NOT NULL
GROUP BY month, country
ORDER BY month, jobs DESC, country;



392 month-country rows; the validation covers all of them, not only the top countries


,month,country,jobs,share_pct
0,2024-02,United States,36127,41.440501
1,2024-02,United Kingdom,6798,7.797839
2,2024-02,India,5924,6.795292
3,2024-02,Australia,4709,5.401592
4,2024-02,Canada,4163,4.775287
5,2024-02,Pakistan,2004,2.298745
6,2024-02,Germany,1755,2.013123
7,2024-02,Netherlands,1647,1.889238


In [11]:
reference = project.read_csv("monthly_country_share.csv")
record("04", compare(q4, reference, ["month", "country"], ["jobs"], ["share_pct"]))

Q04: 784 of 784 compared values pass; largest absolute difference 4.99e-07


'PASS'

## Q5. Jobs per day and hourly share by week

In [12]:
q5 = run_query(database, show_query("05"))
q5

-- Q5. Jobs per day and the hourly share by ISO week (Monday-Sunday).
SELECT iso_week,
       COUNT(DISTINCT posted_date)                       AS days_observed,
       COUNT(*)                                          AS jobs,
       COUNT(*) * 1.0 / COUNT(DISTINCT posted_date)      AS jobs_per_day,
       100.0 * AVG(CASE WHEN contract_type = 'hourly' THEN 1.0 ELSE 0.0 END) AS hourly_share_pct
FROM jobs
WHERE eligible_for_timeseries = 1
GROUP BY iso_week
ORDER BY iso_week;



,iso_week,days_observed,jobs,jobs_per_day,hourly_share_pct
0,2024-W07,3,15497,5165.666667,57.430470
1,2024-W08,7,45543,6506.142857,58.538085
2,2024-W09,7,43644,6234.857143,57.011273
3,2024-W10,7,43719,6245.571429,57.794094
4,2024-W11,7,43008,6144.000000,56.529018
5,2024-W12,6,38109,6351.500000,57.416883


In [13]:
reference = project.read_csv("weekly_market_summary.csv")
record("05", compare(q5, reference, ["iso_week"], ["days_observed", "jobs"], ["jobs_per_day", "hourly_share_pct"]))

Q05: 24 of 24 compared values pass; largest absolute difference 4.29e-07


'PASS'

## Q6. Median hourly offer by client country

SQLite has no MEDIAN function: the query ranks each country's rates with `ROW_NUMBER()` and averages the middle row (odd count) or the two middle rows (even count), the same rule as NumPy. The minimum of jobs per country comes from the configuration.

In [14]:
min_jobs = int(project.settings.section("pay")["min_country_jobs"])
q6 = run_query(database, show_query("06"), {"min_country_jobs": min_jobs})
q6.sort_values("jobs", ascending=False).head(8)

-- Q6. Median hourly offer by client country (Notebook 04's population:
-- jobs eligible for hourly-rate analysis; countries with at least
-- :min_country_jobs such jobs). SQLite has no MEDIAN function, so rows are
-- ranked with a window function: an odd count takes the middle row, an even
-- count averages the two middle rows (the same rule as pandas / NumPy).
WITH population AS (
    SELECT country, hourly_mid
    FROM jobs
    WHERE eligible_for_hourly_rate_analysis = 1
      AND country IS NOT NULL
),
counted AS (
    SELECT country, COUNT(*) AS jobs
    FROM population
    GROUP BY country
    HAVING COUNT(*) >= :min_country_jobs
),
ranked AS (
    SELECT p.country,
           p.hourly_mid,
           c.jobs,
           ROW_NUMBER() OVER (PARTITION BY p.country ORDER BY p.hourly_mid) AS position
    FROM population AS p
    JOIN counted AS c USING (country)
)
SELECT country,
       jobs,
       AVG(hourly_mid) AS median_rate
FROM ranked
WHERE position IN ((jobs + 1) / 2, (jobs + 

,country,jobs,median_rate
87,United States,46620,25.0
86,United Kingdom,8113,22.5
4,Australia,5682,21.0
12,Canada,5054,22.5
33,India,4337,20.0
29,Germany,2022,24.0
55,Netherlands,1554,20.0
60,Pakistan,1379,20.0


In [15]:
reference = project.read_csv("pay_countries.csv")
record("06", compare(q6, reference, ["country"], ["jobs"], ["median_rate"], stored_decimals=None))

Q06: 182 of 182 compared values pass; largest absolute difference 0.00e+00


'PASS'

## Q7. Category labels vs frozen predictions

The query joins the human labels (read from the label sheets) to the frozen predictions and counts the matches itself; it does not read any stored accuracy.

In [16]:
q7 = run_query(database, show_query("07"))
q7

-- Q7. Agreement between the human category labels and the frozen predictions.
-- The blind set is the honest result; the development set tuned the rules.
SELECT l.label_set,
       COUNT(*)                                                           AS labelled,
       SUM(CASE WHEN l.human_category = c.primary_category THEN 1 ELSE 0 END) AS correct,
       AVG(CASE WHEN l.human_category = c.primary_category THEN 1.0 ELSE 0.0 END) AS accuracy
FROM category_labels AS l
JOIN job_categories AS c USING (row_id)
GROUP BY l.label_set
ORDER BY l.label_set;



,label_set,labelled,correct,accuracy
0,blind,350,241,0.688571
1,dev,150,125,0.833333


In [17]:
summary = project.read_csv("label_evaluation_summary.csv")
reference = (summary[(summary["subset"] == "all") & (summary["system"] == "full")]
             .rename(columns={"titles": "labelled"})[["label_set", "labelled", "correct", "accuracy"]])
check = compare(q7.assign(accuracy=q7["accuracy"].round(4)), reference, ["label_set"], ["labelled", "correct"],
                ["accuracy"], stored_decimals=None)
record("07", check)

Q07: 6 of 6 compared values pass; largest absolute difference 0.00e+00


'PASS'

## Q8. Recommender: nDCG@5 and strict Precision@5

Frozen definition (decision D-034): gains 0/1/3 for labels 0/1/2; DCG@5 over each method's ranks 1-5; IDCG@5 from the best five labels of the **whole pooled set** judged for that profile, so every method is compared with the same ideal. The rankings come from the Python recommender; the scores are computed here in SQL.

In [18]:
q8 = run_query(database, show_query("08"))
q8.head(6)

-- Q8. nDCG@5 and strict Precision@5 per profile and method, from the blind
-- relevance judgements (frozen definition, decision D-034):
--   gain = 2^label - 1  (labels 0/1/2 -> gains 0/1/3)
--   DCG@5  = sum over the method's ranks 1-5 of gain / log2(rank + 1)
--   IDCG@5 = the same sum for the best five labels judged for that profile
--            (the whole pooled set, so all methods share one ideal)
--   nDCG@5 = DCG@5 / IDCG@5 (0 when nothing judged is relevant)
--   strict Precision@5 = share of the 5 positions labelled 2
-- Discounts are full-precision constants (SQLite builds may lack log2()).
WITH discount (rank, value) AS (
    VALUES (1, 1.0),
           (2, 0.6309297535714575),
           (3, 0.5),
           (4, 0.4306765580733931),
           (5, 0.3868528072345416)
),
judged AS (
    SELECT r.profile_id, r.method, r.rank, l.relevance,
           CASE l.relevance WHEN 2 THEN 3 WHEN 1 THEN 1 ELSE 0 END AS gain
    FROM recommendation_results AS r
    JOIN relevance_labels

,profile_id,method,ndcg_at_5,strict_precision_at_5
0,P01,keyword,0.773893,0.8
1,P01,tfidf_hybrid,1.000000,1.0
2,P01,tfidf_word,0.886947,0.8
3,P02,keyword,1.000000,1.0
4,P02,tfidf_hybrid,0.902621,0.8
5,P02,tfidf_word,0.912530,0.8


In [19]:
reference = project.read_csv("recommend_relevance_by_profile.csv")
record("08", compare(q8, reference, ["profile_id", "method"], [], ["ndcg_at_5", "strict_precision_at_5"]))

Q08: 72 of 72 compared values pass; largest absolute difference 4.90e-07


'PASS'

In [20]:
# The means per method, also in SQL, against the summary that chose the winner
sql8 = queries["08"].read_text(encoding="utf-8").rstrip().rstrip(";")
means = run_query(database, "SELECT method, AVG(ndcg_at_5) AS mean_ndcg, AVG(strict_precision_at_5) AS mean_precision "
                            f"FROM ({sql8}) GROUP BY method")
reference = project.read_csv("recommend_relevance_summary.csv")
check = compare(means, reference, ["method"], [], ["mean_ndcg", "mean_precision"])
results["08_means"] = check
print(means.round(4))
"PASS" if check["passed"].all() else check[~check["passed"]]

         method  mean_ndcg  mean_precision
0       keyword     0.9527          0.9333
1  tfidf_hybrid     0.9846          0.9667
2    tfidf_word     0.9714          0.9500


'PASS'

## Summary

In [21]:
overview = pd.DataFrame([{"query": number, "compared values": len(table), "passed": int(table["passed"].sum()),
                          "largest difference": table["difference"].abs().max()} for number, table in results.items()])
overview

,query,compared values,passed,largest difference
0,01,2,2,0.000000e+00
1,02,8,8,4.285712e-07
2,03,92,92,4.905425e-07
3,04,784,784,4.989284e-07
4,05,24,24,4.285712e-07
5,06,182,182,0.000000e+00
6,07,6,6,0.000000e+00
7,08,72,72,4.901770e-07
8,08_means,6,6,4.136046e-07


All eight queries reproduce the pipeline: every count is identical, and every decimal agrees to within the 6-decimal rounding of the stored tables (the largest difference is below 0.0000005); the medians and the label accuracy agree exactly. SQL independently reproduces selected aggregations and evaluation metrics from the pipeline's verified row-level outputs. It does not verify the whole project: the cleaned fields, taxonomy predictions and recommender rankings come from Python.

## Claims check

In [22]:
claims = {
    "database fingerprints match this build": matches_build(info, project.manifest),
    "eight query files": len(queries) == 8,
    "all compared values pass": all(table["passed"].all() for table in results.values()),
    "window 37 days, 229,520 jobs": int(q1["days"].iloc[0]) == 37 and int(q1["jobs"].iloc[0]) == 229520,
    "blind: 241 of 350 correct (68.9%)": q7.set_index("label_set").loc["blind", ["labelled", "correct"]].tolist() == [350, 241],
    "hybrid mean nDCG@5 0.985 in SQL": round(means.set_index("method").loc["tfidf_hybrid", "mean_ndcg"], 3) == 0.985,
    "largest decimal difference below 5e-7": max(table["difference"].abs().max() for table in results.values()
                                                if not table["difference"].dropna().empty) < 5e-7 + 1e-9,
    "medians match exactly": results["06"]["difference"].abs().max() == 0,
}
claims = pd.Series(claims, name="true").to_frame()
assert claims["true"].all(), claims[~claims["true"]]
claims

,true
database fingerprints match this build,True
eight query files,True
all compared values pass,True
"window 37 days, 229,520 jobs",True
blind: 241 of 350 correct (68.9%),True
hybrid mean nDCG@5 0.985 in SQL,True
largest decimal difference below 5e-7,True
medians match exactly,True
